# exp122_exact_override_negative_control train

Pilkwang replay の exact-match recovery / guarded overlap override を改善根拠から除外するための negative-control audit。モデル学習と submission 生成は行わない。

## Contents

1. Setup and configuration
2. Input evidence check
3. Run negative-control audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from exact_override_negative_control import run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Root:", paths.root)
print("Experiment dir:", paths.experiment_dir)
print("Artifacts:", paths.artifacts_dir)


## 2. Input evidence check


In [ ]:
def resolve_candidate(raw_path: str) -> Path:
    path = Path(raw_path)
    return path if path.is_absolute() else paths.root / path


audit_cfg = config.get("audit", {})
for key in [
    "notebook_candidates",
    "exp079_summary_json_candidates",
    "exp079_submission_summary_candidates",
    "exp079_pairwise_jsonl_candidates",
    "exp064_metrics_candidates",
    "guard_output_roots",
]:
    print(f"\n{key}")
    for raw in audit_cfg.get(key, []):
        candidate = resolve_candidate(str(raw))
        print("-", candidate, "exists=", candidate.exists())


## 3. Run negative-control audit


In [ ]:
summary = run_audit(paths=paths, config=config)
decision = summary["decision"]
print(json.dumps(decision, indent=2, sort_keys=True))


## 4. Metrics and artifacts


In [ ]:
metrics = json.loads(paths.metrics_path.read_text())
print("metrics_path:", paths.metrics_path)
print("status:", metrics["status"])
print("summary:", metrics["artifacts"]["summary"])
print("notebook_risk_summary:", metrics["artifacts"]["notebook_risk_summary"])
print("guard_output_inventory:", metrics["artifacts"]["guard_output_inventory"])
